In [50]:
######## I HAVE INDICATED WITH !!! THE PARTS WHICH HAVE BEEN ADAPTED FOR TAXICABS


################### IMPORT THE REQUIRED MODULES # pip install dwave-neal

import numpy as np
import neal
import math
import itertools
import sympy as sp
import time

In [51]:
################## CHOOSE THE DWAVE SIMULATED ANNEALING SAMPLER

sampler = neal.SimulatedAnnealingSampler()

In [52]:
################## MAP SPINS {-1,1} TO QUBITS {0,1}

def taufromsig(sig):
    return (sig+1)/2


################### CONVERT FROM BINARY ------> E.G. FROM TAU = [0,1,0] IT RETURNS 0*2^0 + 1*2^1 + 0*2^3 = 2

def Xfromtau(tau): 
    num=0
    for l in range(len(tau)):
        num += tau[l]*2**l
    return int(num)

In [53]:
################## THIS IS TO AUTOMATISE THE REDUCTION OPTIMISING THE NUMBER OF AUXILIARY SPINS. 

################## YOU DO NOT WANT TO TOUCH THIS BOX


def count_couplings(*clists):
    
    occ_list = np.zeros(shape=(0,3))
    
    for l in range(len(clists)):
        
        ncol = clists[l].shape[1]
    
        for i in range(len(clists[l])):
            
            
            minus1 = np.where(clists[l][i][0:ncol-1] == -1)[0]
        
            spins = []
            
            
            if len(minus1) == 0: 
                
                spins = clists[l][i][0:ncol-1]
            
            elif minus1[0] == 2: continue 
        
            else:
                
                
                for j in range(minus1[0]): spins.append(clists[l][i][j])
                    
            
            pairs = list(itertools.combinations(spins, 2)) 
            
            
            for j in range(len(pairs)):
        
                occ = np.where((occ_list[:,0] == pairs[j][0]) & (occ_list[:,1] == pairs[j][1]))[0]

                if len(occ) == 1: occ_list[occ[0]][2] += 1
            
                else: occ_list = np.append(occ_list, np.array([[pairs[j][0], pairs[j][1], 1]]),axis=0)
                    
        
    return occ_list

def replace_pairs(num_spin,most,*higher_coup):
    
    for l in range(len(higher_coup)):
    
        for i in range(len(higher_coup[l])):
            
            ncol = higher_coup[l].shape[1]
            
            minus1 = np.where(higher_coup[l][i][0:ncol-1] == -1)[0]
        
            inds = []
        
            if len(minus1) == 0: 
                
                p_minus1 = ncol-2
                for q in range(ncol-1): inds.append(q)
                

            
            else:
                
                p_minus1 = minus1[0]-1
                for j in range(minus1[0]): inds.append(j)
                    
                
               
            pairs = list(itertools.combinations(inds, 2)) 
            
            
            for j in range(len(pairs)):
                
                i1 = int(pairs[j][0])
                i2 = int(pairs[j][1])
                
    

                if higher_coup[l][i][i1] == most[0] and higher_coup[l][i][i2] == most[1]:
                    
                    higher_coup[l][i][i1] = num_spin
                    higher_coup[l][i][i2] = higher_coup[l][i][p_minus1]
                    higher_coup[l][i][p_minus1] = -1
                    
                    higher_coup[l][i][0:p_minus1].sort()
                    
                    
    return higher_coup

def make_quadratic(num_spin,L,quadratic,linear,*higher_coup):
    
    aux_spin = num_spin-1
    
    while True:
        
        occ_list = count_couplings(*higher_coup)
        
        if len(occ_list) == 0: break
            
        occ_list = occ_list[np.argsort(occ_list[:,2])]
        most = np.array([occ_list[len(occ_list)-1][0],occ_list[len(occ_list)-1][1]])
    
        print("{",int(most[0]),",",int(most[1]),"}")
        aux_spin += 1 
        
        quadratic = np.append(quadratic, np.array([[aux_spin, most[0],-2*L]]),axis=0)
        quadratic = np.append(quadratic, np.array([[aux_spin, most[1],-2*L]]),axis=0)
        linear = np.append(linear, np.array([[aux_spin,3*L]]),axis=0)
            
        occ = np.where((quadratic[:,0] == most[0]) & (quadratic[:,1] == most[1]))[0]
            
        if len(occ) > 0: 
            linear[len(linear)-1][1] += quadratic[occ[0]][2]
            quadratic[occ[0]][2] = L
        
        else:
            quadratic = np.append(quadratic, np.array([[most[0], most[1],L]]),axis=0)
        
        higher_coup = replace_pairs(aux_spin,most,*higher_coup)
    
        occ_list = np.zeros(shape=(0,3))
        
    
    for l in range(len(higher_coup)):
        
        ncol = higher_coup[l].shape[1]
        
        for i in range(len(higher_coup[l])):
            
            if higher_coup[l][i][1] == -1:
                
                occ = np.where(linear[:,0] == higher_coup[l][i][0])[0]
                
                if len(occ) == 0:
                    linear = np.append(linear, np.array([[higher_coup[l][i][0], higher_coup[l][i][ncol-1]]]),axis=0)
                    
                else:
                    linear[occ[0]][1] += higher_coup[l][i][ncol-1]
            else:    
                
                occ = np.where((quadratic[:,0] == higher_coup[l][i][0]) & (quadratic[:,1] == higher_coup[l][i][1]))[0]
                occ1 = np.where((quadratic[:,1] == higher_coup[l][i][0]) & (quadratic[:,0] == higher_coup[l][i][1]))[0]
                
                if len(occ) == 0 and len(occ1) == 0:
                    quadratic = np.append(quadratic, np.array([[higher_coup[l][i][0],higher_coup[l][i][1], higher_coup[l][i][ncol-1]]]),axis=0)
    
                elif len(occ) != 0:
                    quadratic[occ[0]][2] += higher_coup[l][i][ncol-1]
                
                elif len(occ1) != 0:
                    quadratic[occ1[0]][2] += higher_coup[l][i][ncol-1]
    
    print("Number of auxiliary spins: ",aux_spin + 1 - n_var*beta)
    
    
    return quadratic, linear, aux_spin 

In [54]:
################## TEST IF A PROPOSED SOLUTION COMING FROM THE ANNEALER IS A REAL SOLUTION

def itisasolution(lambdas):
    
    z = lambdas[0:n_var]

    return (z[0]**3 + z[1]**3 - z[2]**3 - z[3]**3)**2 ###### !!!

In [55]:
beta = 4      ################## NUMBER OF QUBITS FOR EACH VARIABLE

n_var = 4     ################## !!! YOU NOW NEED 4 VARIABLES 

shift = 1     ################## !!! YOU NEED TO PUT SHIFT = 1 OTHERWISE YOUR BINARY ENCODING GOES FROM 0 TO 15 AND YOU WILL MISS THE SECOND TAXICAB NUMBER

max_exp = 6   ################## !!! THE LARGEST EXPONENT THAT APPEARS IN THE HAMILTONIAN

In [56]:
################## EQS IS A LIST CONTAINING THE EQUATION(S) WE WANT TO SOLVE IN TERM OF THE SYMPY SYMBOLS Z

eqs = []

z = sp.symarray("z",n_var)


eq = (z[0]**3 + z[1]**3 - z[2]**3 - z[3]**3)**2 ###### !!!
eqs.append(eq)


In [57]:
################## THIS IS JUST TO CHOOSE, IF NEEDED, A SUBSET OF THE EQUATIONS/VARIABLES INVOLVED.

################## YOU CAN INGORE IT

eq_chosen = []
var_chosen = []

n_eq = len(eqs)

var_chosen= [n for n in range(n_var)]
eq_chosen = [n for n in range(n_eq)]

In [58]:
################## BINARY ENCODING

sym = sp.symarray("x", n_var*beta)                       
sigmas = np.array([2**n for n in range(beta)])

In [59]:
################## variables IS A LIST CONTAINING THE BINARY REPRESENTATION OF z[0] and z[1]

variables = []

for i in range(n_var): 
    variables.append(np.tensordot(sigmas, sym[i*beta:(i+1)*beta], axes=(0,0)) + shift) 
    
print(variables)

[x_0 + 2*x_1 + 4*x_2 + 8*x_3 + 1, x_4 + 2*x_5 + 4*x_6 + 8*x_7 + 1, 4*x_10 + 8*x_11 + x_8 + 2*x_9 + 1, x_12 + 2*x_13 + 4*x_14 + 8*x_15 + 1]


In [60]:
################## CONSTRUCT THE HAMILTONIAN AS A SYMPY POLYNOMIAL 


sq = 0
for i in range(n_eq): sq += eqs[eq_chosen[i]]
print("The Hamiltonian is: \n\n",sq)
squares = sq.copy()


for i in range(n_var): sq = sq.subs(z[i],variables[i])  ############ IN THE HAMILTONIAN, WE SUBSTITUTE Z WITH ITS BINARY REPRESENTATION
    

################################# !!! NOW WE ADD THE ADDIONAL CONSTRAINTS
constr1 = 1
constr2 = 1


for i in range(beta):
    
    constr1 *= (1 - (sym[i]-sym[2*beta+i])**2) 
    constr2 *= (1 - (sym[i]-sym[3*beta+i])**2) 
    
sq += constr1 + constr2
    
#################################
    
    
    
    
    
    
start = time.time()
H = sp.poly((sq),*sym)       
print("\n\nHamiltonian written in terms of taus:", time.time()-start)

The Hamiltonian is: 

 (z_0**3 + z_1**3 - z_2**3 - z_3**3)**2


Hamiltonian written in terms of taus: 1.3461050987243652


In [61]:
##################### REPLACE TAU^N WITH TAU AS TAU = {0,1}

print("\nReplace powers...")



lsym = [n for n in range(n_var*beta)]
mapping = {sym[i]**j: sym[i] for i,j in list(itertools.product(lsym, [n for n in range(max_exp+1)]))}

start = time.time()
H = H.xreplace(mapping)
print("Done in", time.time()-start)
        
H = sp.poly(H,*sym)

print(H)


Replace powers...
Done in 1.3519260883331299
Poly(16*x_0*x_1*x_2*x_3*x_8*x_9*x_10*x_11 - 8*x_0*x_1*x_2*x_3*x_8*x_9*x_10 - 8*x_0*x_1*x_2*x_3*x_8*x_9*x_11 + 4*x_0*x_1*x_2*x_3*x_8*x_9 - 8*x_0*x_1*x_2*x_3*x_8*x_10*x_11 + 4*x_0*x_1*x_2*x_3*x_8*x_10 + 4*x_0*x_1*x_2*x_3*x_8*x_11 - 2*x_0*x_1*x_2*x_3*x_8 - 8*x_0*x_1*x_2*x_3*x_9*x_10*x_11 + 4*x_0*x_1*x_2*x_3*x_9*x_10 + 4*x_0*x_1*x_2*x_3*x_9*x_11 - 2*x_0*x_1*x_2*x_3*x_9 + 4*x_0*x_1*x_2*x_3*x_10*x_11 - 2*x_0*x_1*x_2*x_3*x_10 - 2*x_0*x_1*x_2*x_3*x_11 + 16*x_0*x_1*x_2*x_3*x_12*x_13*x_14*x_15 - 8*x_0*x_1*x_2*x_3*x_12*x_13*x_14 - 8*x_0*x_1*x_2*x_3*x_12*x_13*x_15 + 4*x_0*x_1*x_2*x_3*x_12*x_13 - 8*x_0*x_1*x_2*x_3*x_12*x_14*x_15 + 4*x_0*x_1*x_2*x_3*x_12*x_14 + 4*x_0*x_1*x_2*x_3*x_12*x_15 - 2*x_0*x_1*x_2*x_3*x_12 - 8*x_0*x_1*x_2*x_3*x_13*x_14*x_15 + 4*x_0*x_1*x_2*x_3*x_13*x_14 + 4*x_0*x_1*x_2*x_3*x_13*x_15 - 2*x_0*x_1*x_2*x_3*x_13 + 4*x_0*x_1*x_2*x_3*x_14*x_15 - 2*x_0*x_1*x_2*x_3*x_14 - 2*x_0*x_1*x_2*x_3*x_15 + 1827842*x_0*x_1*x_2*x_3 + 4608*x_0*x_1*x_2*

In [62]:
################## EXTRACT THE COUPLINGS AS A DICTIONARY, YOU DO NOT NEED TO EDIT THIS

dict_coup = {}

coup_list = list(H.as_dict().items())

for i in range(len(coup_list)):
    
    spins = np.where(np.array(coup_list[i][0][:]) == 1)[0]
    
    if len(spins) > 0:
        
        spins = np.append(spins,coup_list[i][1])
        
        if len(spins)-1 in dict_coup:
            
            dict_coup[len(spins)-1] = np.append(dict_coup[len(spins)-1],np.array([spins]),axis=0)
             
        else:
            
            dict_coup[len(spins)-1] = np.zeros(shape = (0,len(spins)))
            dict_coup[len(spins)-1] = np.append(dict_coup[len(spins)-1],np.array([spins]),axis=0)
            
quad = dict_coup[2]
lin = dict_coup[1]

del dict_coup[1]
del dict_coup[2]

In [63]:
################## AUTOMATIC REDUCTION OF THE HAMILTONIAN

L = 2 * int(max(list(H.as_dict().values())))

reduced = make_quadratic(n_var*beta,L,quad,lin,*list(dict_coup.values()))

{ 0 , 1 }
{ 2 , 3 }
{ 13 , 14 }
{ 9 , 10 }
{ 12 , 15 }
{ 8 , 11 }
{ 4 , 7 }
{ 5 , 6 }
{ 1 , 2 }
{ 0 , 3 }
{ 3 , 16 }
{ 0 , 2 }
{ 0 , 17 }
{ 1 , 17 }
{ 1 , 3 }
{ 2 , 16 }
{ 14 , 20 }
{ 8 , 9 }
{ 15 , 18 }
{ 8 , 19 }
{ 10 , 21 }
{ 14 , 15 }
{ 13 , 20 }
{ 8 , 10 }
{ 12 , 18 }
{ 9 , 11 }
{ 9 , 21 }
{ 13 , 15 }
{ 11 , 19 }
{ 12 , 14 }
{ 12 , 13 }
{ 10 , 11 }
{ 6 , 7 }
{ 5 , 7 }
{ 4 , 6 }
{ 4 , 5 }
{ 5 , 22 }
{ 4 , 23 }
{ 6 , 22 }
{ 7 , 23 }
{ 16 , 17 }
{ 18 , 20 }
{ 19 , 21 }
Number of auxiliary spins:  43


In [64]:
################## CONSTRUCT h AND J

quad = reduced[0].copy()
lin = reduced[1].copy()
aux_spin = reduced[2]

J = np.zeros(shape=(aux_spin+1,aux_spin+1))
h = np.zeros(shape=(aux_spin+1))
     
for i in range(len(quad)):
    
    qi0 = int(quad[i][0])
    qi1 = int(quad[i][1])
    qi2 = quad[i][2]
        
    J[qi0][qi1] = qi2/4
        
    h[qi0] += qi2/4
    h[qi1] += qi2/4
        
for i in range(len(lin)):
    
    li0 = int(lin[i][0])
    li1 = lin[i][1]
        
    h[li0] += li1/2

In [65]:
nreads = 2000           # NUMBER OF ANNEAL RUNS     

answer = sampler.sample_ising(h, J, num_reads=nreads)         # SIMULATED ANNEALING

print(answer)           # PRINT THE OUTPUT


      0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 ... 58       energy num_oc.
51   -1 -1 -1 +1 -1 +1 +1 +1 +1 +1 +1 +1 +1 -1 -1 ... +1 -353107330.0       1
165  +1 -1 -1 -1 +1 +1 +1 +1 -1 -1 -1 +1 -1 +1 +1 ... -1 -353107330.0       1
186  +1 +1 +1 +1 +1 -1 -1 -1 -1 +1 +1 +1 -1 -1 -1 ... -1 -353107330.0       1
249  +1 +1 -1 +1 -1 -1 -1 -1 -1 -1 -1 +1 +1 -1 -1 ... -1 -353107330.0       1
287  -1 -1 -1 +1 +1 -1 -1 +1 +1 +1 -1 +1 -1 -1 -1 ... -1 -353107330.0       1
300  -1 -1 -1 +1 -1 +1 +1 +1 +1 -1 -1 -1 +1 +1 +1 ... -1 -353107330.0       1
438  -1 -1 -1 +1 +1 -1 -1 +1 -1 -1 -1 -1 +1 +1 -1 ... -1 -353107330.0       1
546  +1 +1 -1 +1 -1 -1 -1 -1 +1 -1 -1 +1 -1 -1 -1 ... -1 -353107330.0       1
566  -1 -1 -1 +1 +1 -1 -1 +1 +1 +1 -1 +1 -1 -1 -1 ... -1 -353107330.0       1
576  +1 -1 -1 +1 -1 -1 -1 +1 -1 -1 -1 -1 +1 +1 -1 ... -1 -353107330.0       1
581  +1 -1 -1 -1 +1 +1 +1 +1 -1 +1 +1 +1 -1 -1 -1 ... -1 -353107330.0       1
729  +1 -1 -1 -1 +1 +1 +1 +1 -1 +1 +1 +1 -1 -1 -1 ... -1 -353107

In [66]:
################### THIS IS JUST TO CONVERT FROM SPIN VALUES TO NUMBERS


Lambdasig = []
Lambda = []

for i in range(n_var): 
    
    Lambdasig.append(np.zeros(shape=(nreads,beta)))
    Lambda.append(np.zeros(shape=(nreads)))

values = np.zeros(shape=(nreads))

for i in range(nreads):
    for j in range(beta):
        for k in range(n_var):
        
            Lambdasig[k][i][j] = answer.record[i][0][j+k*beta]
        
    for k in range(n_var):

        Lambda[k][i] = Xfromtau(taufromsig(Lambdasig[k][i])) + shift
        
    current_Lambda = list(zip(*Lambda))[i]
    
    values[i] = itisasolution(current_Lambda)
    
    
    if values[i] == 0:  # IF IT IS A SOLUTION
        if answer.record[i][1] == answer.first.energy: # AND IF IT IS ACTUALLY THE MINIMUM STATE FOUND
            print("\n\n\nSolution:",i,current_Lambda) # THEN ---> PRINT THE NUMBERS 
                
         




Solution: 51 (9.0, 15.0, 16.0, 2.0)



Solution: 165 (2.0, 16.0, 9.0, 15.0)



Solution: 186 (16.0, 2.0, 15.0, 9.0)



Solution: 249 (12.0, 1.0, 9.0, 10.0)



Solution: 287 (9.0, 10.0, 12.0, 1.0)



Solution: 300 (9.0, 15.0, 2.0, 16.0)



Solution: 438 (9.0, 10.0, 1.0, 12.0)



Solution: 546 (12.0, 1.0, 10.0, 9.0)



Solution: 566 (9.0, 10.0, 12.0, 1.0)



Solution: 576 (10.0, 9.0, 1.0, 12.0)



Solution: 581 (2.0, 16.0, 15.0, 9.0)



Solution: 729 (2.0, 16.0, 15.0, 9.0)



Solution: 856 (16.0, 2.0, 15.0, 9.0)



Solution: 896 (15.0, 9.0, 16.0, 2.0)



Solution: 1011 (10.0, 9.0, 1.0, 12.0)



Solution: 1100 (15.0, 9.0, 2.0, 16.0)



Solution: 1285 (15.0, 9.0, 16.0, 2.0)



Solution: 1319 (9.0, 10.0, 12.0, 1.0)



Solution: 1373 (15.0, 9.0, 2.0, 16.0)



Solution: 1414 (9.0, 15.0, 16.0, 2.0)



Solution: 1489 (2.0, 16.0, 9.0, 15.0)



Solution: 1566 (16.0, 2.0, 9.0, 15.0)



Solution: 1729 (12.0, 1.0, 9.0, 10.0)



Solution: 1782 (9.0, 15.0, 2.0, 16.0)



Solution: 1797 (15.0, 9.0, 1